# Week 8: LLM ทำงานอย่างไร (internals)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aofphy/SCI193611_ARTIFICIAL_INTELLIGENCE/blob/main/labs/w08_llm_internals.ipynb)

**Objective:** แกะกลไกสามอย่างที่ทำให้ LLM ทำงานได้ ด้วยโค้ดที่รันเองได้ทั้งหมด

1. การแบ่งโทเคน และเหตุผลที่ภาษาไทยแพงกว่าภาษาอังกฤษ
2. Scaled dot-product attention จากศูนย์
3. การสุ่มโทเคน: อุณหภูมิและ top-p

ส่วนที่ 1 ถึง 3 รันได้ทันทีด้วย `numpy` เท่านั้น ส่วนที่ 4 และ 5 เป็นทางเลือก

## 1) ทำไมภาษาไทยถึงแพงกว่า

โมเดลคิดเงินและนับหน้าต่างบริบทเป็น **โทเคน** ไม่ใช่ตัวอักษร
BPE สมัยใหม่ทำงานบนไบต์ UTF-8 และอักษรไทยหนึ่งตัวใช้ **3 ไบต์**
ขณะที่อักษรละตินใช้ 1 ไบต์ ลองวัดดู

In [ ]:
TH = "ปัญญาประดิษฐ์คือสาขาหนึ่งของวิทยาการคอมพิวเตอร์ที่ศึกษาการสร้างเครื่องจักรที่คิดได้"
EN = "Artificial intelligence is a branch of computer science that studies building machines that think"

for name, s in [("ไทย", TH), ("อังกฤษ", EN)]:
    print(f"{name:8s} chars={len(s):4d}  utf8-bytes={len(s.encode()):4d}  "
          f"bytes/char={len(s.encode())/len(s):.2f}")

# ประโยคสองประโยคนี้มีความหมายเดียวกัน แต่จำนวนไบต์ต่างกันมาก
# ซึ่งเป็นตัวขับหลักของจำนวนโทเคนใน tokenizer แบบ byte-level BPE

### BPE ขนาดจิ๋ว

Byte-Pair Encoding เริ่มจากหน่วยเล็กที่สุด แล้ว **รวมคู่ที่พบบ่อยที่สุด** ซ้ำไปเรื่อย ๆ
โค้ดข้างล่างคือ BPE ทั้งอัลกอริทึม ใน 15 บรรทัด

In [ ]:
from collections import Counter

def train_bpe(corpus, n_merges):
    """เรียนรู้กฎการรวมคู่จาก corpus คืนรายการ merge ตามลำดับที่เรียนรู้"""
    words = [list(w) + ["</w>"] for w in corpus.split()]
    merges = []
    for _ in range(n_merges):
        pairs = Counter()
        for w in words:
            pairs.update(zip(w, w[1:]))
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        merges.append(best)
        words = [_apply(w, best) for w in words]
    return merges, words

def _apply(word, pair):
    out, i = [], 0
    while i < len(word):
        if i + 1 < len(word) and (word[i], word[i + 1]) == pair:
            out.append(word[i] + word[i + 1]); i += 2
        else:
            out.append(word[i]); i += 1
    return out

corpus = ("low low low low low lower lower newest newest newest "
          "newest newest newest widest widest widest")
merges, words = train_bpe(corpus, 8)
print("merges:", merges)
print("tokens:", words[0], words[5], words[7])

# self-check: การรวมคู่ต้องทำให้จำนวนโทเคนรวมลดลงเสมอ
before = sum(len(list(w)) + 1 for w in corpus.split())
after = sum(len(w) for w in words)
assert after < before, (before, after)
print(f"OK: {before} -> {after} โทเคน")

## 2) Attention จากศูนย์

$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-1, -2) / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)   # -inf ในตำแหน่งที่ห้ามมอง
    A = softmax(scores)
    return A @ V, A

def multi_head_attention(X, Wq, Wk, Wv, Wo, n_heads, causal=True):
    """คืนทั้ง output และเมทริกซ์ attention ต่อหัว (n_heads, T, T) เพื่อให้ตรวจสอบได้"""
    T, d = X.shape
    dh = d // n_heads
    split = lambda t: t.reshape(T, n_heads, dh).transpose(1, 0, 2)
    mask = np.tril(np.ones((T, T), bool)) if causal else None
    out, A = attention(split(X @ Wq), split(X @ Wk), split(X @ Wv), mask)
    return out.transpose(1, 0, 2).reshape(T, d) @ Wo, A

In [ ]:
np.random.seed(0)
T, d, H = 5, 8, 2
X = np.random.randn(T, d)
Wq, Wk, Wv, Wo = (np.random.randn(d, d) for _ in range(4))

y, A = multi_head_attention(X, Wq, Wk, Wv, Wo, H)
print("MHA output shape:", y.shape, " attention ต่อหัว:", A.shape)

# self-check: ยิงใส่เส้นทางที่ใช้จริง ไม่ใช่ attention() เปล่า ๆ
assert np.allclose(A.sum(-1), 1.0), "แต่ละแถวของ attention ต้องรวมได้ 1"
assert np.allclose(np.triu(A, 1), 0), "ห้ามมองอนาคต (ตรวจทุกหัว)"
y_full, _ = multi_head_attention(X, Wq, Wk, Wv, Wo, H, causal=False)
assert not np.allclose(y, y_full), "ปิด causal แล้วผลต้องเปลี่ยน ไม่งั้นแปลว่า mask ไม่ถูกใช้"
print("OK: causal mask ทำงานถูกต้องทุกหัว")

print("\nหัวที่ 1 (แถว i = โทเคน i แจกความสนใจให้ใคร):")
print(np.round(A[1], 3))

### วิธีอ่านเมทริกซ์ attention

แต่ละแถวรวมได้ 1 และสามเหลี่ยมบนเป็น 0 คือหลักฐานว่า causal mask ทำงาน

- **แถว 0 ได้ `[1, 0, 0, 0, 0]` เสมอ** ไม่ว่าคะแนนจะเป็นเท่าไร เพราะ softmax ของค่าเดียวได้ 1 แถวนี้จึงไม่ได้บอกอะไรเรื่องเนื้อหา
- softmax สนใจแค่ **ผลต่าง** ของคะแนน ผลต่าง 0.65 ให้สัดส่วน $e^{0.65}\approx 1.9$ เท่า ส่วนผลต่างเกิน 5 ยุบเหลือเกือบ one-hot ทันที
- นี่คือเหตุผลที่ต้องหาร $\sqrt{d_k}$ ถ้าไม่หาร คะแนนจะโตตาม $d$ แล้ว softmax อิ่มตัว เกรเดียนต์หาย

**กับดักที่ต้องระวัง:** ถ้าลองเรียก `attention(X, X, X)` ตรง ๆ โดยไม่มี projection จะได้เมทริกซ์ที่เกือบเป็น identity เพราะ $s_{ii}=\lVert x_i\rVert^2$ และตาม Cauchy-Schwarz ค่าสูงสุดของทั้งเมทริกซ์ต้องอยู่บนเส้นทแยงเสมอ นั่นเป็นผลของการที่ $Q=K$ ไม่ใช่พฤติกรรมของ attention จริง ของจริงมี $W_q \neq W_k$ ซึ่งหักความสมมาตรทิ้ง แต่ละหัวจึงเรียนรู้รูปแบบการมองย้อนหลังคนละแบบได้

## 3) การสุ่มโทเคน: อุณหภูมิและ top-p

โมเดลคืนค่า logits แล้วเราเป็นคนเลือกว่าจะแปลงเป็นโทเคนอย่างไร
สองพารามิเตอร์นี้คือสิ่งที่คุณจะปรับผ่าน API ในสัปดาห์หน้า

In [ ]:
def sample(logits, temperature=1.0, top_p=1.0, rng=np.random):
    """เลือกดัชนีโทเคนหนึ่งตัวจาก logits"""
    if temperature <= 0:                       # greedy
        return int(np.argmax(logits))
    p = softmax(np.asarray(logits, float) / temperature)
    order = np.argsort(-p)
    keep = np.cumsum(p[order]) <= top_p
    keep[0] = True                             # เก็บตัวที่น่าจะเป็นสูงสุดไว้เสมอ
    idx = order[keep]
    return int(rng.choice(idx, p=p[idx] / p[idx].sum()))

vocab = ["แมว", "หมา", "นก", "ปลา", "เต่า"]
logits = np.array([4.0, 3.0, 1.0, 0.5, -2.0])

for T_ in (0.0, 0.3, 1.0, 2.0):
    rng = np.random.default_rng(0)
    draws = [vocab[sample(logits, T_, rng=rng)] for _ in range(20)]
    print(f"T={T_:<4} {Counter(draws).most_common()}")

# self-check
assert sample(logits, 0.0) == 0, "T=0 ต้องเลือกตัวที่ logit สูงสุดเสมอ"
rng = np.random.default_rng(1)
assert all(sample(logits, 1.0, top_p=0.5, rng=rng) in (0, 1) for _ in range(50)), \
    "top_p=0.5 ต้องตัดหางที่ไม่น่าจะเป็นออก"
print("OK")

## 4) (ทางเลือก) tokenizer จริง

ต้องมี `transformers` ติดตั้ง: `pip install transformers`
เปรียบเทียบจำนวนโทเคนจริงของภาษาไทยกับภาษาอังกฤษ

In [ ]:
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
    for name, s in [("ไทย", TH), ("อังกฤษ", EN)]:
        ids = tok.encode(s)
        print(f"{name:8s} tokens={len(ids):3d}  chars/token={len(s)/len(ids):.2f}")
    print("\nโทเคน 12 ตัวแรกของประโยคภาษาไทย:")
    print([tok.decode([i]) for i in tok.encode(TH)[:12]])
except Exception as e:
    print("ข้ามส่วนนี้:", type(e).__name__, e)

## 5) (ทางเลือก) เรียกโมเดลจริงแล้วขยับอุณหภูมิ

ทฤษฎีเรื่องการสุ่มโทเคนในข้อ 4 จะชัดขึ้นมากเมื่อเห็นโมเดลจริงเปลี่ยนพฤติกรรม
เลือกทางใดทางหนึ่ง

* **บนเครื่องตัวเอง** ติดตั้ง [Ollama](https://ollama.com) แล้ว `ollama pull qwen3:8b`
  ไม่ต้องมี key ไม่ส่งข้อมูลออกนอกเครื่อง
* **ผ่าน API ฟรี** ตั้ง `OPENROUTER_API_KEY` แล้วเลือกโมเดล `:free`

**ทางเลือกที่ไม่เสียเงิน** สมัคร [openrouter.ai](https://openrouter.ai/) เอา key ใส่
`OPENROUTER_API_KEY` แล้วใช้โมเดลที่ลงท้ายด้วย `:free` ดูรายชื่อที่ใช้ได้ตอนนี้ด้วย
`python llm.py --free` ข้อแลกเปลี่ยนคือมีเพดานคำขอต่อนาทีและต่อวัน
และคิวอาจยาวช่วงคนใช้เยอะ

การเทียบอุณหภูมิต้องใช้ **โมเดลเดิม** ทุกครั้ง ถ้าใช้ `openrouter/free`
ระบบจะสุ่มโมเดลให้ใหม่ทุกคำขอ ผลที่ได้จึงเทียบกันไม่ได้
ให้ตรึงชื่อโมเดลด้วย `LLM_MODEL` ก่อน เช่น

```bash
export LLM_MODEL=google/gemma-4-31b-it:free
```


In [ ]:
try:                                  # ไคลเอนต์กลางของแล็บสัปดาห์ 8 ถึง 14
    import llm as api
except ImportError:                   # บน Colab ที่มีแต่ไฟล์สมุดบันทึก ให้ดึงมาก่อน
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/aofphy/"
        "SCI193611_ARTIFICIAL_INTELLIGENCE/main/labs/llm.py", "llm.py")
    import llm as api

print(api.describe(api.resolve()))

for t in (0.0, 1.3):
    print(f"\n--- temperature={t} ---")
    try:
        print(api.chat("ตั้งชื่อโครงงาน AI สำหรับนักศึกษาฟิสิกส์ มา 3 ชื่อ",
                       temperature=t, max_tokens=200))
    except Exception as e:
        print("ยังต่อโมเดลไม่ได้:", type(e).__name__, e)
        break


## TODO และการส่งงาน

**TODO**
1. เพิ่ม positional encoding แบบ sinusoidal แล้วดูว่าผลของ attention เปลี่ยนไปอย่างไร
2. ต่อ feed-forward, residual และ LayerNorm ให้ครบเป็น Transformer block
3. เขียนใหม่ด้วย `torch.nn.MultiheadAttention` แล้วเทียบตัวเลขกับของคุณเอง
4. วัดว่า `train_bpe` ต้องใช้กี่ merge จึงจะรวมคำภาษาไทยที่พบบ่อยเป็นโทเคนเดียวได้

**ส่งงาน:** notebook ที่รันครบทุกเซลล์ พร้อมตอบคำถาม
"อุณหภูมิเปลี่ยนผลลัพธ์ของโมเดลอย่างไร และคุณจะเลือกค่าเท่าไรสำหรับงานสกัดข้อมูล เพราะอะไร"